# Chạy tiếp: luật GFlowNet từ image_concept — swin_t — seeds 42/44

Khôi phục CBM từ lần chạy trước (không train lại Baseline/CBM/RF/GFlowNet cũ), sau đó chạy **Stage 4C** (GFlowNet xây luật `concept ∧ ¬concept ⇒ class`) và **Stage 5** ở các mode `penalty`, `head`, `both` trên 2 GPU song song.

Kết quả: `outputs_seed_*/04c_concept_rules/`, `05c_concept_*/`, `concept_rules_comparison.csv` và `concept_rules_outputs.zip`.

In [ ]:
# =========================================================================
# THAM SỐ CHẠY TIẾP (sinh tự động bởi generate_concept_rules_kaggle_notebook.py)
# =========================================================================
DATA_DIR = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/khmer'
CONCEPT_ANNOTATIONS_PATH = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/image_concepts.json'
NUM_CLASSES = 9
N_CONCEPTS = 54
BACKBONE = 'swin_t'
SEED_A = 42
SEED_B = 44
CONCEPT_WEIGHT = 0.5
BATCH_SIZE = 64
NUM_WORKERS = 2
NUM_EPOCHS = 50
PATIENCE = 5
MODES = ['penalty', 'head', 'both']          # penalty | head | both
GIT_REPO = 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git'
GIT_BRANCH = 'newgfn'


In [ ]:
# Kiểm tra dữ liệu ảnh và file nhãn concept (tự tìm lại nếu đường dẫn khác)
from pathlib import Path

data_path = Path(DATA_DIR)
concept_path = Path(CONCEPT_ANNOTATIONS_PATH)
if not data_path.exists():
    candidates = list(Path("/kaggle/input").glob("**/khmer"))
    if candidates:
        DATA_DIR = str(candidates[0])
        print(f"Tự động phát hiện DATA_DIR: {DATA_DIR}")
    else:
        raise FileNotFoundError(f"Không tìm thấy DATA_DIR: {DATA_DIR}")
if not concept_path.exists():
    candidates = list(Path("/kaggle/input").glob("**/*concept*.json"))
    if candidates:
        CONCEPT_ANNOTATIONS_PATH = str(candidates[0])
        print(f"Tự động phát hiện file concept: {CONCEPT_ANNOTATIONS_PATH}")
    else:
        raise FileNotFoundError(f"Không tìm thấy file concept: {CONCEPT_ANNOTATIONS_PATH}")
print("DATA_DIR:", DATA_DIR)
print("CONCEPT_ANNOTATIONS_PATH:", CONCEPT_ANNOTATIONS_PATH)


In [ ]:
# Clone repo đúng branch và kiểm tra branch ĐÃ CÓ mã luật concept (patch đã được push)
import subprocess
from pathlib import Path

PROJECT = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(PROJECT)], check=True)

required = [
    "src/gflownet/concept_rule_pipeline.py",
    "src/models/rule_head.py",
    "pipelines/stage4_concept_rules_gflownet.py",
    "pipelines/run_concept_rules_pipeline.py",
]
missing = [p for p in required if not (PROJECT / p).exists()]
if missing:
    raise RuntimeError(
        f"Branch '{GIT_BRANCH}' chưa có mã luật concept (thiếu {missing}). "
        "Hãy commit & push patch lên branch trước khi chạy notebook."
    )
print("Repo sẵn sàng tại:", PROJECT, "| đã có mã luật concept")


In [ ]:
# Cài đặt thư viện (giữ nguyên numpy của môi trường Kaggle)
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
import importlib, importlib.metadata as md, subprocess, sys

print("numpy gốc:", md.version("numpy"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchgfn", "tensordict", "openpyxl", "psutil"], check=True)
importlib.invalidate_caches()
for name in tuple(sys.modules):
    if name == "gfn" or name.startswith("gfn."):
        del sys.modules[name]

import torch
n_gpus = torch.cuda.device_count()
print("Số GPU:", n_gpus)
assert n_gpus >= 2, "HÃY CHỌN ACCELERATOR: GPU T4 x2 trên Kaggle!"
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {p.name} | VRAM {p.total_memory / 1024**3:.1f} GB")


In [ ]:
# Khôi phục kết quả lần chạy trước từ /kaggle/input (thư mục hoặc file zip).
import json, shutil, zipfile
from pathlib import Path

REPO = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
INPUT = Path("/kaggle/input")
EXTRACT_ROOT = Path("/kaggle/working/prev_extract")


def _matches(seed_dir: Path) -> bool:
    meta_path = seed_dir / "cbm" / "cbm_metadata.json"
    if not meta_path.exists():
        return False
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return meta.get("architecture") == BACKBONE and int(meta.get("n_concepts", -1)) == N_CONCEPTS


def find_previous(seed: int) -> Path | None:
    for cand in INPUT.glob(f"**/outputs_seed_{seed}"):
        if cand.is_dir() and _matches(cand):
            return cand
    # Không thấy thư mục: thử giải nén các zip kết quả rồi tìm lại
    if not EXTRACT_ROOT.exists():
        for z in INPUT.glob("**/*outputs*.zip"):
            target = EXTRACT_ROOT / z.stem
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)
    for cand in EXTRACT_ROOT.glob(f"**/outputs_seed_{seed}"):
        if cand.is_dir() and _matches(cand):
            return cand
    return None


restored = {}
for seed in (SEED_A, SEED_B):
    src = find_previous(seed)
    if src is None:
        raise FileNotFoundError(
            f"Không tìm thấy outputs_seed_{seed} của backbone '{BACKBONE}' ({N_CONCEPTS} concept) "
            "trong /kaggle/input. Hãy attach dataset kết quả lần chạy trước."
        )
    dst = REPO / f"outputs_seed_{seed}"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    restored[seed] = dst
    print(f"Seed {seed}: khôi phục từ {src} -> {dst}")


In [ ]:
# Cập nhật params.yaml theo notebook (đường dẫn, backbone, concept, batch/worker chống OOM)
import yaml

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)
params["data_dir"] = DATA_DIR
params["num_classes"] = NUM_CLASSES
params["num_epochs"] = NUM_EPOCHS
params["patience"] = PATIENCE
params["batch_size"] = BATCH_SIZE
params["num_workers"] = NUM_WORKERS
params["baseline_comparison"]["selected_architecture"] = BACKBONE
params.setdefault("cbm", {})
params["cbm"]["n_concepts"] = N_CONCEPTS
params["cbm"]["concept_weight"] = CONCEPT_WEIGHT
params["cbm"]["concept_annotations_path"] = CONCEPT_ANNOTATIONS_PATH
with open("params.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(params, f, sort_keys=False, allow_unicode=True)
print("Đã cập nhật params.yaml")


In [ ]:
# Stage 4C + Stage 5 (penalty, head, both) cho 2 seed song song trên 2 GPU
import subprocess, sys
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
subprocess.run(
    [sys.executable, "-m", "pipelines.run_concept_rules_dual_gpu",
     "--config", "params.yaml", "--seeds", str(SEED_A), str(SEED_B), "--modes", *MODES],
    check=True,
)


In [ ]:
# Bảng so sánh: mô hình cũ (đã train) và mô hình mới (luật từ image_concept)
import pandas as pd
from IPython.display import display

for seed in (SEED_A, SEED_B):
    csv = REPO / f"outputs_seed_{seed}" / "concept_rules_comparison.csv"
    print(f"--- Seed {seed} ---")
    display(pd.read_csv(csv))


In [ ]:
# Đóng gói luật mới và bảng so sánh để tải về
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!zip -r -q /kaggle/working/concept_rules_outputs.zip outputs_seed_*/04c_concept_rules outputs_seed_*/05c_concept_* outputs_seed_*/concept_rules_comparison.csv outputs_seed_*/execution_concept_rules_*.log
!ls -lh /kaggle/working/concept_rules_outputs.zip
